# Re-run: existence and direction (Paper II, RQ1)

This notebook recomputes the headline numbers of RQ1 from what release `learning-graph-budget-2026`
ships: the committed outputs in its code package, `code/studies/.../out/*.json`, and, for the power
calculation, the package's own `positivity.mde`. It needs Python 3.9+ with numpy, and the unzipped
release (download it from the report, then run this notebook from the release folder or its
`notebooks/` folder, or set `RELEASE_DIR` to it).

**What it can and cannot recompute.** The atlas itself, which places every learner of every pair in a
cell, reads the corpora (ASSISTments 2012–13, EDM Cup, EdNet, XES3G5M, Eedi, Duolingo SLAM and the
DataShop exports), and none of them may be redistributed; `reproduce.py` reruns it for a reader who
has obtained them (steps `rq1-atlas`, `rq1-empty-cell`). What the release keeps is each corpus's
**funnel**: how many pairs reached and passed each condition of a verdict (coverage, cell, readability,
power, the reachability gate) and how many supported pairs survived the ability check. From those
counts this notebook recomputes every share Table 1 and the text print, rather than re-reading the
printed shares. It recomputes the group size a 0.10 drop needs from each corpus's median control
accuracy with the atlas's own MDE function, the lag decay of Table 2 from each corpus's band means,
and the orientation sweep's identity with a difficulty sort from its 30 cells.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "learning-graph-budget-2026"   # filled in by board/render_report.py
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Code package:", shown(code_dir()))

Code package: code


In [2]:
import json, math, sys
import numpy as np

empty = load_code_json("studies/identifiability/empty-cell/out/result.json")["corpora"]
junyi = load_code_json("studies/identifiability/empty-cell/out/junyi-exercise.json")["reading"]
rows = {"ASSISTments 2012–13": empty["assist"], "EDM Cup 2023": empty["edmcup"],
        "Junyi (ds1275), exercises": junyi, "Duolingo SLAM en_es": empty["slam-en_es-construct"],
        "EdNet KT1": empty["ednet"], "XES3G5M": empty["xes3g5m"], "Eedi": empty["eedi"]}

def table1(r):
    f = r["funnel"]
    reached, passed = f["reached"], f["passed"]
    pairs = reached["COVERAGE"]
    decided = passed["GATE"]                       # supported + contradicted, after the reachability gate
    return {"learners": r["learners"], "pairs": pairs, "in_common": r["coverage_median_N"],
            "cell": passed["CELL"], "readable": passed["READABILITY"], "powered": passed["POWER"],
            "supported": f["pre_gate"]["supported"], "contradicted": decided - f["pre_gate"]["supported"],
            "decided_pct": 100 * decided / pairs,
            "powered_pct_of_readable": 100 * passed["POWER"] / passed["READABILITY"] if passed["READABILITY"] else None,
            "survive": f["ability"]["survive"], "tested": f["ability"]["supported"]}

T = {k: table1(r) for k, r in rows.items()}
print(f"{'Corpus':28s} {'pairs':>6s} {'common':>6s} {'cell':>6s} {'read':>6s} {'power':>5s} {'sup/con':>10s} {'decided':>8s}")
for k, t in T.items():
    print(f"{k:28s} {t['pairs']:6,d} {t['in_common']:6.0f} {t['cell']:6,d} {t['readable']:6,d} {t['powered']:5,d} "
          f"{t['supported']:>5,d}/{t['contradicted']:<4,d} {t['decided_pct']:7.3f}%")

Corpus                        pairs common   cell   read power    sup/con  decided
ASSISTments 2012–13          40,000      4  9,940  9,350   133    87/0      0.217%
EDM Cup 2023                 20,000      0    648    509     6     3/0      0.015%
Junyi (ds1275), exercises    20,000      1     38     18     0     0/0      0.000%
Duolingo SLAM en_es          20,000     25  5,253  5,000   125    19/0      0.095%
EdNet KT1                    19,740    415 16,202 16,201   236   119/9      0.648%
XES3G5M                      20,000     12  5,073  4,851 1,148   736/69     4.025%
Eedi                         20,000    420 13,243 13,120 5,797 4,935/138   25.365%


In [3]:
# Table 1 as printed: the decided share (EDM Cup to three decimals), the counts, and the median number
# of learners in common. Shares are computed from the counts above and rounded once.
PRINTED = {  # corpus: (learners, pairs, in common, cell >= 5, readable, powered, supported, contradicted, decided %)
    "ASSISTments 2012–13": (29018, 40000, 4, 9940, 9350, 133, 87, 0, "0.22"),
    "EDM Cup 2023": (34580, 20000, 0, 648, 509, 6, 3, 0, "0.015"),
    "Junyi (ds1275), exercises": (2063, 20000, 1, 38, 18, 0, 0, 0, "0.00"),
    "Duolingo SLAM en_es": (2593, 20000, 25, 5253, 5000, 125, 19, 0, "0.10"),
    "EdNet KT1": (6000, 19740, 415, 16202, 16201, 236, 119, 9, "0.65"),
    "XES3G5M": (18066, 20000, 12, 5073, 4851, 1148, 736, 69, "4.03"),
    "Eedi": (118971, 20000, 420, 13243, 13120, 5797, 4935, 138, "25.36"),
}
for k, (n, p, com, cell, read, pw, sup, con, dec) in PRINTED.items():
    t = T[k]
    got = (t["learners"], t["pairs"], round(t["in_common"]), t["cell"], t["readable"], t["powered"],
           t["supported"], t["contradicted"])
    assert got == (n, p, com, cell, read, pw, sup, con), (k, got)
    nd = len(dec.split(".")[1])
    assert f"{t['decided_pct']:.{nd}f}" == dec, (k, t["decided_pct"], dec)
print("Table 1: 56 counts and 7 decided shares match the paper.")

Table 1: 56 counts and 7 decided shares match the paper.


In [4]:
# Power is the condition with the lowest pass rate: the share of readable pairs that are powered.
pw = {k: t["powered_pct_of_readable"] for k, t in T.items() if t["powered_pct_of_readable"] is not None}
for k, v in pw.items():
    print(f"{k:28s} powered {v:5.1f}% of readable pairs")
small = ["ASSISTments 2012–13", "EDM Cup 2023", "Duolingo SLAM en_es", "EdNet KT1"]
assert f"{max(pw[k] for k in small):.1f}" == "2.5"          # "at most 2.5% are powered on ASSISTments, EDM Cup, SLAM and EdNet"
assert f"{pw['XES3G5M']:.1f}" == "23.7" and f"{pw['Eedi']:.1f}" == "44.2"
assert 100 * T["Eedi"]["powered"] / T["Eedi"]["pairs"] > 25      # "a powered test for more than a quarter of the pairs scored"
for k, r in rows.items():   # the binding condition is the stage with the lowest pass rate, POWER in each
    f = r["funnel"]
    rates = {s: f["passed"][s] / f["reached"][s] for s in ("CELL", "READABILITY", "POWER") if f["reached"][s]}
    print(f"{k:28s} lowest pass rate at {min(rates, key=rates.get)}")
print("Powered shares match the paper.")

ASSISTments 2012–13          powered   1.4% of readable pairs
EDM Cup 2023                 powered   1.2% of readable pairs
Junyi (ds1275), exercises    powered   0.0% of readable pairs
Duolingo SLAM en_es          powered   2.5% of readable pairs
EdNet KT1                    powered   1.5% of readable pairs
XES3G5M                      powered  23.7% of readable pairs
Eedi                         powered  44.2% of readable pairs
ASSISTments 2012–13          lowest pass rate at POWER
EDM Cup 2023                 lowest pass rate at POWER
Junyi (ds1275), exercises    lowest pass rate at POWER
Duolingo SLAM en_es          lowest pass rate at POWER
EdNet KT1                    lowest pass rate at POWER
XES3G5M                      lowest pass rate at POWER
Eedi                         lowest pass rate at POWER
Powered shares match the paper.


## The ability check (Table 1's last column, and the abstract's "at most 27.6%")

A supported pair survives when its drop still reaches its minimum detectable effect once each learner's
accuracy on other skills is taken out. The share is survivors over supported pairs, in the four
corpora with at least 20 supported pairs.

In [5]:
surv = {k: (t["survive"], t["tested"]) for k, t in T.items() if t["tested"] >= 20}
for k, (s, n) in surv.items():
    print(f"{k:28s} {s:4d} of {n:5,d} survive = {100 * s / n:.1f}%")
PRINTED_SURV = {"Eedi": (684, 4935, "13.9"), "XES3G5M": (99, 736, "13.5"), "EdNet KT1": (11, 119, "9.2"),
                "ASSISTments 2012–13": (24, 87, "27.6")}
assert set(surv) == set(PRINTED_SURV), surv
for k, (s, n, pct) in PRINTED_SURV.items():
    assert surv[k] == (s, n) and f"{100 * s / n:.1f}" == pct, (k, surv[k], pct)
assert f"{max(100 * s / n for s, n in surv.values()):.1f}" == "27.6"
print("Surviving shares match the paper; the largest is 27.6%.")

ASSISTments 2012–13            24 of    87 survive = 27.6%
EdNet KT1                      11 of   119 survive = 9.2%
XES3G5M                        99 of   736 survive = 13.5%
Eedi                          684 of 4,935 survive = 13.9%
Surviving shares match the paper; the largest is 27.6%.


## How many learners a test of a 0.10 drop needs

The atlas's minimum detectable effect is `positivity.mde(n_control, n_treat, p_control)`: the smallest
drop an 80%-powered two-sided 5% test of two proportions can detect, solved numerically because the
treated arm's variance depends on the drop. The size two equal groups need is the smallest *n* with
`mde(n, n, p) <= 0.10`, at each corpus's median accuracy in the control cell. The function is imported
from the code package, the code that ran.

In [6]:
sys.path.insert(0, str(code_dir() / "studies/identifiability/overlap"))
sys.path.insert(0, str(code_dir()))
from positivity import mde, Z_ALPHA, Z_BETA, MIN_CELL   # noqa: E402

def required_n(p, d=0.10):
    lo, hi = MIN_CELL, 200000
    while lo < hi:
        mid = (lo + hi) // 2
        v = mde(mid, mid, p)
        lo, hi = (lo, mid) if v is not None and v <= d else (mid + 1, hi)
    return lo

need = {}
for k, r in list(empty.items()) + [("datashop/ds1275-exercise", junyi)]:
    p = r["funnel"]["size"]["median_control_accuracy"]
    if p is None or not 0.01 < p < 0.99:
        continue
    need[k] = required_n(p)
    assert need[k] == r["funnel"]["size"]["per_group_n_for_mde_0.10"], (k, need[k])
    print(f"{k:28s} median control accuracy {p:.3f}: {need[k]} learners per group")
main = [k for k in need if k != "datashop/ds1275-exercise"]   # the corpora of the atlas's main reading
assert (min(need[k] for k in main), max(need[k] for k in main)) == (277, 379)
print("Between 277 and 379 learners in each group, as printed (Section 5.1, the abstract and the answer).")
# the readable median identifying cells, against that need
for k, key in [("ASSISTments", "assist"), ("EDM Cup", "edmcup"), ("SLAM en_es", "slam-en_es-construct"),
               ("EdNet", "ednet"), ("XES3G5M", "xes3g5m"), ("Eedi", "eedi")]:
    print(f"   {k:12s} median identifying cell {empty[key]['funnel']['size']['median_identifying_cell_readable']:.0f}")
assert [round(empty[k]["funnel"]["size"]["median_identifying_cell_readable"]) for k in
        ("assist", "edmcup", "slam-en_es-construct", "ednet", "xes3g5m", "eedi")] == [21, 19, 19, 43, 50, 178]

datashop/ds580               median control accuracy 0.472: 379 learners per group
datashop/ds671               median control accuracy 0.636: 377 learners per group
datashop/ds649               median control accuracy 0.707: 351 learners per group
datashop/ds1190              median control accuracy 0.689: 359 learners per group
datashop/ds1706              median control accuracy 0.803: 289 learners per group
datashop/ds1275              median control accuracy 0.429: 366 learners per group
xes3g5m                      median control accuracy 0.817: 277 learners per group
ednet                        median control accuracy 0.646: 375 learners per group
edmcup                       median control accuracy 0.680: 362 learners per group
eedi                         median control accuracy 0.631: 379 learners per group
assist                       median control accuracy 0.653: 372 learners per group
slam-en_es-construct         median control accuracy 0.809: 283 learners per group
slam

## Most negative verdicts fail on power, not on positivity

Over the atlas file's corpora (`overlap/out/atlas.json`: EDM Cup, EdNet, XES3G5M, Eedi and the DataShop
sets that were read), the pairwise nulls that were powered, the ones the reachability gate returned to
unknown, and those whose merge node was readable but not powered, summed corpus by corpus.

In [7]:
atlas = load_code_json("studies/identifiability/overlap/out/atlas.json")
read = [r for r in atlas if r.get("contradicted_before_reachability") is not None]   # skipped sets carry no counts
before = sum(r["contradicted_before_reachability"] for r in read)
down = sum(r["downgraded_unfalsifiable"] for r in read)
unpowered = sum(r["downgraded_readable_but_unpowered"] for r in read)
print(f"{len(read)} corpora: {before:,} powered nulls, {down:,} returned to unknown, {unpowered:,} with a readable, "
      f"unpowered merge node ({100 * unpowered / down:.1f}%)")
assert (f"{before:,}", f"{down:,}", f"{unpowered:,}", f"{100 * unpowered / down:.1f}") == ("1,399", "1,183", "1,155", "97.6")
assert next(r for r in atlas if r["dataset"] == "eedi")["learners"] == 118971

12 corpora: 1,399 powered nulls, 1,183 returned to unknown, 1,155 with a readable, unpowered merge node (97.6%)


## The drop by lag band (Table 2)

D is the mean drop at 7 days or more minus the mean drop within a day, over pairs readable in both
bands. The relative decay is −D divided by the drop within a day. The intervals come from a cluster
bootstrap over dependent skills in the study; this notebook does not redraw them.

In [8]:
lag = load_code_json("studies/identifiability/overlap/out/lag_bands.json")["corpora"]
PRINTED_LAG = {"assist": (644, "+0.158", "+0.096", "−0.062", "39.0"), "xes3g5m": (175, "+0.153", "+0.120", "−0.033", "21.7"),
               "ednet": (6170, "+0.058", "+0.043", "−0.015", None), "eedi": (4643, "+0.166", "+0.118", "−0.048", None)}
sg = lambda x, d=3: f"{x:+.{d}f}".replace("-", "−")
decay = {}
for k, (n, early, late, D, pct) in PRINTED_LAG.items():
    p = lag[k]["primary"]
    d = p["late_mean"] - p["early_mean"]              # recomputed, not read from mean_D
    assert abs(d - p["mean_D"]) < 1e-12
    decay[k] = -d / p["early_mean"]
    print(f"{k:8s} {p['pairs_both']:5,d} pairs  {sg(p['early_mean'])} -> {sg(p['late_mean'])}  D {sg(d)}  "
          f"smaller by {100 * decay[k]:.1f}%  [{p['verdict']}]")
    assert (p["pairs_both"], sg(p["early_mean"]), sg(p["late_mean"]), sg(d)) == (n, early, late, D), k
    assert d < 0 and p["verdict"] == "DECAYS"         # the sign: smaller at long lags
assert f"{100 * min(decay.values()):.1f}" == "21.7" and f"{100 * max(decay.values()):.1f}" == "39.0"
assert min(decay, key=decay.get) == "xes3g5m" and max(decay, key=decay.get) == "assist"
assert lag["edmcup"]["primary"]["pairs_both"] == 0
print("Table 2 and the decay range (21.7% on XES3G5M to 39.0% on ASSISTments) match the paper.")

assist     644 pairs  +0.158 -> +0.096  D −0.062  smaller by 39.0%  [DECAYS]
xes3g5m    175 pairs  +0.153 -> +0.120  D −0.033  smaller by 21.7%  [DECAYS]
ednet    6,170 pairs  +0.058 -> +0.043  D −0.015  smaller by 25.3%  [DECAYS]
eedi     4,643 pairs  +0.166 -> +0.118  D −0.048  smaller by 28.8%  [DECAYS]
Table 2 and the decay range (21.7% on XES3G5M to 39.0% on ASSISTments) match the paper.


## The orientation statistic is a difficulty sort

With failure rates *f*ₐ and *f*ᵦ and the rate *j* of failing both, s(*a*, *b*) = *j*/*f*ₐ − *j*/*f*ᵦ
= *j*(*f*ᵦ − *f*ₐ)/(*f*ₐ*f*ᵦ), so whenever *j* > 0 its sign is the sign of *f*ᵦ − *f*ₐ. The first cell
checks the identity on random rates; the second checks, in every one of the sweep's 30 cells, that the
statistic's accuracy equals the difficulty sort's, and recomputes the accuracies the text prints.

In [9]:
rng = np.random.default_rng(1)
fa, fb = rng.uniform(0.01, 0.99, 100000), rng.uniform(0.01, 0.99, 100000)
j = rng.uniform(0, 1, 100000) * np.minimum(fa, fb)
s = j / fa - j / fb
keep = j > 0
assert np.array_equal(np.sign(s[keep]), np.sign((fb - fa)[keep]))
print("sign(s) = sign(f_b - f_a) on 100,000 random triples with j > 0")

cells = load_code_json("studies/simulation/orientation-sortability/out/sim.json")["cells"]
n = 0
for mech, cs in cells.items():
    for name, c in cs.items():
        assert c["asym"]["acc"] == c["diffsort"]["acc"], (mech, name)
        n += 1
assert n == 30
print(f"In all {n} cells the statistic's accuracy equals the difficulty sort's.")
acc = lambda m, c: cells[m][c]["asym"]
f3 = lambda a: f"{a['acc']:.3f} [{a['lo']:.3f}, {a['hi']:.3f}]"
PRINTED_SWEEP = {("AND", "INDEPENDENT/1.0"): "0.883 [0.854, 0.910]", ("AND", "INDEPENDENT/0.0"): "0.773 [0.736, 0.809]",
                 ("AND", "INVERTED/1.0"): "0.772 [0.721, 0.814]", ("AND", "INVERTED/0.0"): "0.520 [0.462, 0.580]"}
for (m, c), want in PRINTED_SWEEP.items():
    print(f"{m} {c:16s} {f3(acc(m, c))}")
    assert f3(acc(m, c)) == want, (m, c)
aligned = [cells["AND"][f"ALIGNED/{h}"]["asym"]["acc"] for h in ("1.0", "0.75", "0.5", "0.25", "0.0")]
assert (f"{min(aligned):.3f}", f"{max(aligned):.3f}") == ("0.998", "1.000")
for regime, lo_hi in (("ALIGNED", ("0.969", "0.975")), ("INDEPENDENT", ("0.551", "0.558")), ("INVERTED", ("0.069", "0.098"))):
    v = [cells["OR"][f"{regime}/{h}"]["asym"]["acc"] for h in ("1.0", "0.75", "0.5", "0.25", "0.0")]
    assert (f"{min(v):.3f}", f"{max(v):.3f}") == lo_hi, (regime, min(v), max(v))
print("The sweep's printed accuracies match: conjunctive 0.998 to 1.000 where difficulty rises with depth; "
      "disjunctive 0.969-0.975, 0.551-0.558 and 0.069-0.098.")

sign(s) = sign(f_b - f_a) on 100,000 random triples with j > 0
In all 30 cells the statistic's accuracy equals the difficulty sort's.
AND INDEPENDENT/1.0  0.883 [0.854, 0.910]
AND INDEPENDENT/0.0  0.773 [0.736, 0.809]
AND INVERTED/1.0     0.772 [0.721, 0.814]
AND INVERTED/0.0     0.520 [0.462, 0.580]
The sweep's printed accuracies match: conjunctive 0.998 to 1.000 where difficulty rises with depth; disjunctive 0.969-0.975, 0.551-0.558 and 0.069-0.098.


## Placing the real corpora

On Junyi's authored graph, S is the share of eligible edges whose dependent is harder; the committed
value is a ratio of counts over the 233 eligible edges, which the cell recovers. On ASSISTments it is
the share of cross-grade pairs whose higher-grade skill is harder.

In [10]:
real = load_code_json("studies/simulation/orientation-sortability/out/real.json")
jy, asst = real["junyi2015"], real["assistments2012"]["all_matched"]
k = round(jy["S"][0] * jy["n_eligible_edges"])
assert k / jy["n_eligible_edges"] == jy["S"][0]
print(f"Junyi: dependent harder on {k} of {jy['n_eligible_edges']} eligible edges = {k / jy['n_eligible_edges']:.3f} "
      f"[{jy['S'][1]:.3f}, {jy['S'][2]:.3f}]; rho {jy['rho'][0]:+.3f}; {jy['verdict']}")
assert f"{k / jy['n_eligible_edges']:.3f}" == "0.579" and (f"{jy['S'][1]:.3f}", f"{jy['S'][2]:.3f}") == ("0.515", "0.639")
assert sg(jy["rho"][0]) == "+0.476" and jy["floor"]["passes"] is False
print(f"ASSISTments: higher grade harder in {asst['S'][0]:.3f} of {asst['n_cross_grade_pairs']:,} pairs, rho {sg(asst['rho'][0])}")
assert (f"{asst['S'][0]:.3f}", asst["n_cross_grade_pairs"], asst["n_skills"], sg(asst["rho"][0])) == ("0.576", 6871, 131, "+0.187")
print("The placements match the paper.")

Junyi: dependent harder on 135 of 233 eligible edges = 0.579 [0.515, 0.639]; rho +0.476; DIFFICULTY RISES WITH DEPTH (ALIGNED SIDE)
ASSISTments: higher grade harder in 0.576 of 6,871 pairs, rho +0.187
The placements match the paper.


## With the corpus: ASSISTments 2012–13's learners and skills (optional)

Everything above reads only the release. The atlas's own inputs need the corpora. This last cell reads
one of them if the reader has it: ASSISTments 2012–13, as `assist_rich.csv`, which
`code/data/ingest/assist2012/rich.py` derives from the deposited `assist2012.zip` (location and hash in
the release file; `REPRODUCE.md` names the variable `ASSIST_DIR`). It counts the learners and skills
the atlas reads, rows with a skill only, and checks them against Table 1 (29,018 learners) and the
lag-band study (265 skills). Without the file it says so and stops; the rest of the notebook does not
depend on it. It streams about 500 MB, so it takes about a minute.

In [11]:
import csv
try:
    path = corpus_file("ASSIST_DIR", "assist_rich.csv")
except FileNotFoundError as exc:
    print("Skipped, the corpus is not on this machine:", exc)
else:
    learners, skills = set(), set()
    with open(path, newline="", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            if r["skill_id"] and r["skill_id"] != "NA":
                learners.add(r["user_id"])
                skills.add(r["skill_id"])
    print(f"{len(learners):,} learners and {len(skills)} skills with a skill-tagged attempt")
    assert len(learners) == T["ASSISTments 2012–13"]["learners"] == lag["assist"]["learners"] == 29018
    assert len(skills) == lag["assist"]["items"] == 265
    print("The corpus gives the learners and skills the committed outputs record.")

Skipped, the corpus is not on this machine: This notebook also needs assist_rich.csv from the public corpus. Download the corpus (its location and hash are in the release file), then set ASSIST_DIR to its folder or put assist_rich.csv beside this notebook.
